# Phase 6 DAEAC FCBA RR Late-Fusion NSV DS1 Oversampled Pretrain

Runs two DS1-oversampled pretrain options: sqrt class weight and no class weight.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import numpy as np
import pandas as pd
os.environ['PYTHONUNBUFFERED'] = '1'

REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'
CONFIGS = [
    ('ds1_os_sqrtw', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_ds1_os_sqrtw.yaml', 'daeac_fcba_rr_nsv_ds1_os_sqrtw_base_best'),
    ('ds1_os_noweight', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_ds1_os_noweight.yaml', 'daeac_fcba_rr_nsv_ds1_os_noweight_base_best'),
]

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'

## 1. Clone Repo and Install

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
else:
    print('Repo already exists:', ECG)
assert ECG.exists(), ECG
os.chdir(ECG)
!pip install -q -r requirements.txt
!python scripts/check_repo.py

## 2. Copy NSV RR Bundle With DS1 Oversampled

In [ ]:
candidate_dirs = [p.parent for p in Path('/kaggle/input').glob('**/record_split_manifest.json')]
print('Candidate record-split bundle dirs:')
for p in candidate_dirs:
    print(' -', p)
assert len(candidate_dirs) == 1, f'Expected one NSV bundle, found {candidate_dirs}'

DEST = Path('data/processed/phase6_daeac_record_splits_nsv')
DEST.mkdir(parents=True, exist_ok=True)
for src in sorted(candidate_dirs[0].glob('*.npz')):
    shutil.copy2(src, DEST / src.name)
manifest = candidate_dirs[0] / 'record_split_manifest.json'
if manifest.exists():
    shutil.copy2(manifest, DEST / manifest.name)

required = ('ds1_train_oversampled.npz', 'ds1_val.npz', 'ds2_test.npz')
for name in required:
    path = DEST / name
    assert path.exists(), f'Missing {path}. Attach the bundle containing ds1_train_oversampled.npz.'
    with np.load(path, allow_pickle=True) as data:
        assert data['class_names'].tolist() == ['N', 'S', 'V'], (name, data['class_names'].tolist())
        assert 'rr_features' in data.files, f'{name} has no rr_features.'
        assert data['rr_features'].shape == (len(data['x']), 7), (name, data['rr_features'].shape)

with np.load(DEST / 'ds1_train_oversampled.npz', allow_pickle=True) as data:
    labels, counts = np.unique(data['y'], return_counts=True)
    print('ds1_train_oversampled counts:', dict(zip(labels.tolist(), counts.tolist())))
    print('aug_total:', int(data['is_augmented'].sum()))
!ls -lh data/processed/phase6_daeac_record_splits_nsv | head

## 3. Validate Both Configs

In [ ]:
for run_id, cfg, _ in CONFIGS:
    print('\nVALIDATE', run_id)
    subprocess.run([sys.executable, 'scripts/phase6_daeac_paper/00_validate_data.py', '--config', cfg], check=True)

## 4. Run Two Pretrains and DS2 Evaluations

In [ ]:
rows = []
for run_id, cfg, method_name in CONFIGS:
    print('\n' + '=' * 80)
    print('RUN', run_id)
    print('=' * 80)
    subprocess.run([sys.executable, '-u', 'scripts/phase6_daeac_paper/01_train_base.py', '--config', cfg], check=True)

    # Read output_dir from the loaded YAML through the training summary path glob.
    candidates = sorted(Path('outputs').glob(f'phase6_daeac_fcba_latefusion_rr_nsv_{run_id.replace("ds1_os_", "ds1_os_")}/metrics/daeac_base_train_summary.json'))
    if not candidates:
        candidates = sorted(Path('outputs').glob(f'*{run_id}*/metrics/daeac_base_train_summary.json'))
    assert len(candidates) == 1, candidates
    summary_path = candidates[0]
    summary = json.loads(summary_path.read_text())
    best_checkpoint = Path(summary['best_checkpoint'])
    assert best_checkpoint.exists(), best_checkpoint

    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/03_eval.py',
        '--config', cfg,
        '--checkpoint', str(best_checkpoint),
        '--method-name', method_name,
        '--dataset', 'target',
    ], check=True)

    metrics_path = summary_path.parents[1] / 'metrics' / f'{method_name}_target_test_metrics.json'
    metrics = json.loads(metrics_path.read_text())
    cm = np.asarray(metrics['confusion_matrix'])
    rows.append({
        'run_id': run_id,
        'best_epoch': summary['best_epoch'],
        'best_val_macro_f1': summary['best_val_macro_f1'],
        'test_accuracy': metrics['accuracy'],
        'test_macro_f1': metrics['macro_f1'],
        'N_f1': metrics['per_class']['N']['f1'],
        'S_precision': metrics['per_class']['S']['precision'],
        'S_recall': metrics['per_class']['S']['recall'],
        'S_f1': metrics['per_class']['S']['f1'],
        'V_f1': metrics['per_class']['V']['f1'],
        'N_to_S': int(cm[0, 1]),
        'pred_S': int(cm[:, 1].sum()),
        'summary_path': str(summary_path),
        'metrics_path': str(metrics_path),
    })

df = pd.DataFrame(rows).sort_values('test_macro_f1', ascending=False)
display(df)
out = Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_ds1_os_two_run_summary.csv')
df.to_csv(out, index=False)
print('Saved', out)

## 5. Package Outputs

In [ ]:
!zip -r /kaggle/working/phase6_daeac_fcba_latefusion_rr_nsv_ds1_os_two_run_outputs.zip outputs/phase6_daeac_fcba_latefusion_rr_nsv_ds1_os_* outputs/phase6_daeac_fcba_latefusion_rr_nsv_ds1_os_two_run_summary.csv